# Word embedding analogies

In [ ]:
# I use Windows VS Code with Python and Jupyter Notebook. Data-loading and preprocessing are adapted with the help of OpenAI Codex.
%matplotlib inline

import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "8")
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("MPLBACKEND", "Agg")
import concurrent.futures
import gzip
import hashlib
import html
import json
import math
from pathlib import Path
import platform
import random
import re
import shutil
import sys
import time
import urllib.parse
import urllib.request
import zipfile

import gensim
from gensim.models import Word2Vec, KeyedVectors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
from threadpoolctl import threadpool_limits

ROOT = Path.cwd()
CACHE = ROOT / "data"
CACHE.mkdir(exist_ok=True)
RESULTS = CACHE / "results"
RESULTS.mkdir(exist_ok=True)
SEED = 42
WIKI_IDS = [
    "1jiu9E1NalT2Y8EIuWNa1xf2Tw1f1XuGd",
    "1ABblLRd9HXdXvaNv8H9fFq984bhnowoG",
    "1z2VFNhpPvCejTP5zyejzKj5YjI_Bn42M",
    "1VKjded9BxADRhIoCzXy_W8uzVOTWIf0g",
    "16mBeG26m9LzHXdPe8UrijUIc6sHxhknz",
    "17JFvxOH-kc-VmvGkhG7p3iSZSpsWdgJI",
    "19IvB2vOJRGlrYulnTXlZECR8zT5v550P",
    "1sjwO8A2SDOKruv6-8NEq7pEIuQ50ygVV",
    "1s7xKWJmyk98Jbq6Fi1scrHy7fr_ellUX",
    "17eQXcrvY1cfpKelLbP2BhQKrljnFNykr",
    "1J5TAN6bNBiSgTIYiPwzmABvGhAF58h62",
]


def download(url, destination):
    """Download atomically, resuming partial transfers when supported."""
    destination = Path(destination)
    if destination.is_file() and destination.stat().st_size:
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_suffix(destination.suffix + ".partial")
    for attempt in range(4):
        try:
            request_url = url
            if "drive.google.com" in url:
                with urllib.request.urlopen(url, timeout=60) as response:
                    if "text/html" in response.headers.get("Content-Type", ""):
                        page = response.read().decode("utf-8")
                        action = re.search(r'<form[^>]+action="([^"]+)"', page)
                        if action is None:
                            raise RuntimeError("Google Drive did not expose a download form")
                        params = dict(re.findall(
                            r'<input[^>]+name="([^"]+)"[^>]+value="([^"]*)"', page))
                        request_url = html.unescape(action.group(1)) + "?" + urllib.parse.urlencode(params)
            offset = partial.stat().st_size if partial.exists() else 0
            request = urllib.request.Request(request_url, headers={"Range": f"bytes={offset}-"} if offset else {})
            with urllib.request.urlopen(request, timeout=120) as response:
                if "text/html" in response.headers.get("Content-Type", ""):
                    raise RuntimeError("Received HTML instead of corpus/model data")
                append = offset > 0 and response.status == 206
                expected = response.headers.get("Content-Length")
                received = 0
                started = time.monotonic()
                last_message = started
                with partial.open("ab" if append else "wb") as output:
                    while chunk := response.read(4 * 1024 * 1024):
                        output.write(chunk)
                        received += len(chunk)
                        if time.monotonic() - last_message > 45:
                            print(f"Downloading {destination.name}: {received / 1e6:.0f} MB", flush=True)
                            last_message = time.monotonic()
                if expected is not None and received != int(expected):
                    raise IOError(f"Incomplete transfer: {received} of {expected} bytes")
            partial.replace(destination)
            print(f"Downloaded {destination.name}: {destination.stat().st_size / 1e6:.1f} MB", flush=True)
            return destination
        except Exception:
            if attempt == 3:
                raise
            time.sleep(2 ** attempt)


def download_wiki(indices=range(11)):
    def one(index):
        return download("https://drive.google.com/uc?export=download&id=" + WIKI_IDS[index],
                        CACHE / f"wiki_texts_part_{index}.txt.gz")
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
        return list(pool.map(one, indices))


def prepare_analogy():
    path = ROOT / "questions-words.txt"
    if not path.exists():
        supplied_csv = ROOT / "questions-words.csv"
        if not supplied_csv.exists():
            return download("https://raw.githubusercontent.com/tmikolov/word2vec/master/questions-words.txt", path)
        table = pd.read_csv(supplied_csv)
        with path.open("w", encoding="utf-8") as output:
            previous = None
            for row in table.itertuples(index=False):
                if row.SubCategory != previous:
                    output.write(row.SubCategory + "\n")
                    previous = row.SubCategory
                output.write(row.Question + "\n")
    return path


def load_pretrained():
    path = CACHE / "glove-wiki-gigaword-100.kv"
    if path.exists():
        return KeyedVectors.load(str(path), mmap="r")
    archive = download(
        "https://github.com/RaRe-Technologies/gensim-data/releases/download/"
        "glove-wiki-gigaword-100/glove-wiki-gigaword-100.gz",
        CACHE / "glove-wiki-gigaword-100.gz")
    vectors = KeyedVectors.load_word2vec_format(str(archive), binary=False)
    vectors.save(str(path))
    return vectors


def install_portable_loaders():
    """Adapt only the template's data-loading operations; keep its cells intact."""
    import gensim.downloader
    original_load = gensim.downloader.load
    def cached_load(name, *args, **kwargs):
        if name == "glove-wiki-gigaword-100" and not args and not kwargs:
            return load_pretrained()
        return original_load(name, *args, **kwargs)
    gensim.downloader.load = cached_load
    shell = get_ipython()
    original_system = shell.system
    def portable_system(command):
        command = command.strip()
        if command.startswith("wget ") and "questions-words.txt" in command:
            print("Analogy text:", prepare_analogy())
        elif command.startswith("gdown ") and "wiki_texts_part_" in command:
            index = int(re.search(r"wiki_texts_part_(\d+)", command).group(1))
            download_wiki([index])
        elif command == "gunzip -k wiki_texts_part_*.gz":
            print("All 11 gzip parts will be decompressed directly into the combined corpus.")
        elif command == "cat wiki_texts_part_*.txt > wiki_texts_combined.txt":
            target = ROOT / "wiki_texts_combined.txt"
            if not target.exists():
                temporary = target.with_suffix(".partial")
                with temporary.open("wb") as output:
                    for index in range(11):
                        with gzip.open(CACHE / f"wiki_texts_part_{index}.txt.gz", "rb") as source:
                            shutil.copyfileobj(source, output, length=8 * 1024 * 1024)
                        print(f"Combined Wikipedia part {index + 1}/11", flush=True)
                temporary.replace(target)
            print(f"Combined corpus: {target.stat().st_size:,} bytes")
        elif command == "head -n 10 wiki_texts_combined.txt":
            with (ROOT / "wiki_texts_combined.txt").open(encoding="utf-8") as source:
                for _, line in zip(range(10), source):
                    print(line.rstrip())
        else:
            return original_system(command)
    shell.system = portable_system


def sample_articles(source, output, fraction, seed=SEED):
    """Uniform, exact-size sampling without replacement using a two-pass stream.

    A common seeded permutation is not held in memory. Sequential selection
    uses remaining_needed / remaining_total, giving every k-subset equal probability.
    """
    source = Path(source)
    output = Path(output)
    count_path = CACHE / "wiki_article_count.json"
    stat = {"bytes": source.stat().st_size, "mtime_ns": source.stat().st_mtime_ns}
    stored = json.loads(count_path.read_text()) if count_path.exists() else {}
    if stored.get("source") != stat:
        print("Counting Wikipedia articles...", flush=True)
        with source.open(encoding="utf-8") as stream:
            total = sum(1 for _ in stream)
        stored = {"source": stat, "articles": total}
        count_path.write_text(json.dumps(stored))
    total = stored["articles"]
    needed = math.floor(total * fraction)
    metadata = {"source": stat, "total_articles": total, "sample_articles": needed,
                "fraction": fraction, "seed": seed}
    manifest = output.with_suffix(".json")
    if output.exists() and manifest.exists() and json.loads(manifest.read_text()) == metadata:
        print(f"Reusing {needed:,}/{total:,} articles ({fraction:.0%})")
        return metadata
    generator = random.Random(seed)
    remaining = needed
    temporary = output.with_suffix(".partial")
    with source.open(encoding="utf-8") as stream, temporary.open("w", encoding="utf-8") as selected:
        for index, line in enumerate(stream):
            if generator.randrange(total - index) < remaining:
                selected.write(line)
                remaining -= 1
    assert remaining == 0
    temporary.replace(output)
    manifest.write_text(json.dumps(metadata, indent=2))
    print(f"Sampled {needed:,}/{total:,} articles ({fraction:.0%})", flush=True)
    return metadata


PARAMETERS = dict(vector_size=100, window=5, min_count=5, sg=0, negative=5,
                  hs=0, sample=1e-3, epochs=5, seed=SEED,
                  workers=min(8, os.cpu_count() or 1), alpha=0.025,
                  min_alpha=0.0001, max_final_vocab=200000)


def prepare_training_text(source, destination):
    """Preserve inflections and stop words; remove non-ASCII alphabetic tokens.

    An article remains a single line; Word2Vec's LineSentence/corpus_file reader
    bounds internal chunks at 10,000 tokens. No artificial article boundaries
    are introduced between distinct articles.
    """
    source, destination = Path(source), Path(destination)
    signature = {"bytes": source.stat().st_size, "mtime_ns": source.stat().st_mtime_ns,
                 "preprocessing": "lowercase_ascii_alpha_v1"}
    manifest = destination.with_suffix(".json")
    if destination.exists() and manifest.exists():
        cached = json.loads(manifest.read_text())
        if cached["signature"] == signature:
            return cached
    articles, tokens, retained_articles = 0, 0, 0
    temporary = destination.with_suffix(".partial")
    with source.open(encoding="utf-8") as stream, temporary.open("w", encoding="utf-8") as output:
        for line in stream:
            articles += 1
            words = [w.lower() for w in line.split() if w.isascii() and w.isalpha()]
            tokens += len(words)
            if words:
                retained_articles += 1
                output.write(" ".join(words) + "\n")
    temporary.replace(destination)
    stats = {"signature": signature, "articles": articles, "retained_articles": retained_articles,
             "tokens": tokens, "bytes": destination.stat().st_size}
    manifest.write_text(json.dumps(stats, indent=2))
    print(f"Preprocessed {articles:,} articles / {tokens:,} tokens", flush=True)
    return stats


def train_embeddings(source, label):
    normalized = CACHE / f"{label}_tokens.txt"
    stats = prepare_training_text(source, normalized)
    checkpoint = CACHE / f"{label}.model"
    metadata_path = CACHE / f"{label}_training.json"
    signature = {"parameters": PARAMETERS, "corpus": stats}
    if checkpoint.exists() and metadata_path.exists():
        metadata = json.loads(metadata_path.read_text())
        if metadata["signature"] == signature:
            print(f"Reusing completed model: {label}", flush=True)
            return Word2Vec.load(str(checkpoint)), metadata
    print(f"Training {label}: {stats['tokens']:,} tokens; {PARAMETERS}", flush=True)
    from gensim.models.callbacks import CallbackAny2Vec
    class Progress(CallbackAny2Vec):
        def __init__(self): self.epoch = 0
        def on_epoch_end(self, model):
            self.epoch += 1
            print(f"{label}: epoch {self.epoch}/{PARAMETERS['epochs']} complete", flush=True)
    started = time.monotonic()
    model = Word2Vec(corpus_file=str(normalized), callbacks=[Progress()], **PARAMETERS)
    model.save(str(checkpoint))
    metadata = {"signature": signature, "vocabulary": len(model.wv),
                "effective_min_count": model.effective_min_count,
                "elapsed_seconds": time.monotonic() - started,
                "corpus_count": model.corpus_count, "corpus_total_words": model.corpus_total_words}
    metadata_path.write_text(json.dumps(metadata, indent=2))
    return model, metadata


def analogy_predictions(vectors, table, label, batch_size=128):
    """Full-vocabulary 3CosAdd search, excluding A/B/C; OOV queries count wrong.

    Equivalent to most_similar(positive=[B,C], negative=[A], topn=1),
    with normalized source vectors and cosine scoring. Never use gold D
    to choose a candidate or restrict the search to benchmark words.
    """
    rows = [tuple(q.lower().split()) for q in table["Question"]]
    assert all(len(row) == 4 for row in rows)
    cache_path = RESULTS / f"{label}_predictions.csv"
    manifest_path = RESULTS / f"{label}_evaluation.json"
    fingerprint = hashlib.sha256()
    fingerprint.update("\n".join(table["Question"]).encode())
    fingerprint.update("\n".join(vectors.index_to_key).encode())
    fingerprint.update(vectors.vectors[::max(1, len(vectors) // 1000)].tobytes())
    signature = {"fingerprint": fingerprint.hexdigest(), "method": "normalized_3cosadd_full_vocabulary_v1"}
    if cache_path.exists() and manifest_path.exists() and json.loads(manifest_path.read_text()) == signature:
        detail = pd.read_csv(cache_path, keep_default_na=False)
        print(f"Reusing verified evaluation: {label}")
        print(summarize_scores(detail).to_string(index=False))
        return detail["Prediction"].tolist(), detail["Gold"].tolist(), detail
    keys = vectors.key_to_index
    predictions = ["<OOV>" for _ in rows]
    valid = [i for i, row in enumerate(rows) if all(word in keys for word in row[:3])]
    covered = [all(word in keys for word in row) for row in rows]
    normalized = vectors.get_normed_vectors()
    with threadpool_limits(limits=8):
        for start in range(0, len(valid), batch_size):
            indices = valid[start:start + batch_size]
            source_ids = np.array([[keys[word] for word in rows[i][:3]] for i in indices])
            queries = normalized[source_ids[:, 1]] + normalized[source_ids[:, 2]] - normalized[source_ids[:, 0]]
            norms = np.linalg.norm(queries, axis=1, keepdims=True)
            queries = queries / np.maximum(norms, np.finfo(np.float32).eps)
            similarities = queries @ normalized.T
            for j in range(3):
                similarities[np.arange(len(indices)), source_ids[:, j]] = -np.inf
            winners = np.argmax(similarities, axis=1)
            for i, winner in zip(indices, winners):
                predictions[i] = vectors.index_to_key[int(winner)]
            if start % (batch_size * 20) == 0:
                print(f"Evaluating {label}: {start:,}/{len(valid):,} in-vocabulary queries", flush=True)
    gold = [row[3] for row in rows]
    detail = table.copy()
    detail["Gold"] = gold
    detail["Prediction"] = predictions
    detail["Covered"] = covered
    detail["Correct"] = np.array(gold) == np.array(predictions)
    detail.to_csv(RESULTS / f"{label}_predictions.csv", index=False)
    scores = summarize_scores(detail)
    scores.to_csv(RESULTS / f"{label}_scores.csv", index=False)
    manifest_path.write_text(json.dumps(signature, indent=2))
    print(scores.to_string(index=False))
    return predictions, gold, detail


def summarize_scores(detail):
    rows = []
    groups = [("Overall", "All", detail)]
    for column in ["Category", "SubCategory"]:
        groups.extend((column, str(name), group) for name, group in detail.groupby(column, sort=False))
    for level, name, group in groups:
        n = len(group)
        coverage = int(group["Covered"].sum())
        correct = int(group["Correct"].sum())
        rows.append(dict(Level=level, Group=name, Questions=n, Covered=coverage, Correct=correct,
                         Accuracy=correct / n, Coverage=coverage / n,
                         CoveredAccuracy=correct / coverage if coverage else float("nan")))
    return pd.DataFrame(rows)


def plot_family(vectors, table, title, filename):
    family = table.loc[table["SubCategory"] == ": family", "Question"]
    words = sorted({word.lower() for question in family for word in question.split()})
    present = [word for word in words if word in vectors]
    missing = sorted(set(words) - set(present))
    print(f"Family words: {len(present)}/{len(words)}; missing: {missing}")
    if len(present) < 3:
        raise ValueError("Need at least three family words for t-SNE")
    features = np.vstack([vectors.get_vector(word, norm=True) for word in present])
    perplexity = min(10, len(present) - 1)
    with threadpool_limits(limits=8):
        coordinates = TSNE(n_components=2, perplexity=perplexity, init="pca",
                           learning_rate="auto", random_state=SEED, max_iter=1500,
                           metric="euclidean").fit_transform(features)
    fig, axis = plt.subplots(figsize=(12, 9))
    axis.scatter(coordinates[:, 0], coordinates[:, 1], s=35, color="#245b87")
    axis.set_title(title)
    axis.set_xlabel("t-SNE dimension 1")
    axis.set_ylabel("t-SNE dimension 2")
    axis.margins(0.14)
    fig.tight_layout()
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    from matplotlib.transforms import Bbox
    point_boxes = [Bbox.from_bounds(px - 4, py - 4, 8, 8)
                   for px, py in axis.transData.transform(coordinates)]
    occupied = []
    offsets = [(4, 4), (4, -12), (4, 16), (4, -24), (-4, 4), (-4, -12),
               (4, 28), (4, -36), (-4, 16), (-4, -24)]
    for point_index, (word, (x, y)) in enumerate(zip(present, coordinates)):
        for dx, dy in offsets:
            annotation = axis.annotate(word, (x, y), xytext=(dx, dy),
                                       textcoords="offset points", fontsize=9,
                                       ha="left" if dx > 0 else "right")
            box = annotation.get_window_extent(renderer).expanded(1.04, 1.12)
            other_points = point_boxes[:point_index] + point_boxes[point_index + 1:]
            if not any(box.overlaps(other) for other in occupied + other_points):
                occupied.append(box)
                if abs(dy) >= 16 or dx < 0:
                    annotation.remove()
                    axis.annotate(word, (x, y), xytext=(dx, dy), textcoords="offset points",
                                  fontsize=9, ha="left" if dx > 0 else "right",
                                  arrowprops=dict(arrowstyle="-", color="#999999", lw=0.5))
                break
            annotation.remove()
        else:
            annotation = axis.annotate(word, (x, y), xytext=(4, 40),
                                       textcoords="offset points", fontsize=9)
            occupied.append(annotation.get_window_extent(renderer))
    fig.savefig(RESULTS / filename, dpi=160, bbox_inches="tight")
    return fig


def prepare_brown():
    archive = download("https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/brown.zip",
                       CACHE / "brown.zip")
    target = CACHE / "brown_articles.txt"
    if not target.exists():
        with zipfile.ZipFile(archive) as corpus, target.open("w", encoding="utf-8") as output:
            documents = sorted(name for name in corpus.namelist() if re.fullmatch(r"brown/c[a-z]\d{2}", name))
            assert len(documents) == 500, len(documents)
            for name in documents:
                content = corpus.read(name).decode("ascii")
                output.write(" ".join(token.rsplit("/", 1)[0] for token in content.split()) + "\n")
    return target


def nearest_words(models, selected=("king", "queen", "computer", "music", "city")):
    rows = []
    for label, vectors in models.items():
        for word in selected:
            if word not in vectors:
                rows.append({"Model": label, "Word": word, "Neighbors (cosine)": "OOV"})
            else:
                neighbors = vectors.most_similar(word, topn=5)
                rows.append({"Model": label, "Word": word,
                             "Neighbors (cosine)": "; ".join(f"{w} ({s:.3f})" for w, s in neighbors)})
    result = pd.DataFrame(rows)
    result.to_csv(RESULTS / "nearest_words.csv", index=False)
    return result

install_portable_loaders()


## Part I: Data Pre-processing

In [ ]:
import pandas as pd

In [ ]:
# Download the Google Analogy dataset
!wget http://download.tensorflow.org/data/questions-words.txt

In [ ]:
# Preprocess the dataset
file_name = "questions-words"
with open(f"{file_name}.txt", "r") as f:
    data = f.read().splitlines()

In [ ]:
# check data from the first 10 entries
for entry in data[:10]:
    print(entry)

In [ ]:
# TODO1: Write your code here for processing data to pd.DataFrame
# Please note that the first five mentions of ": " indicate `semantic`,
# and the remaining nine belong to the `syntatic` category.

# I implement the data-processing with the help of OpenAI Codex.
questions, categories, sub_categories = [], [], []
header_count = 0
sub_category = None
for line_number, entry in enumerate(data, start=1):
    entry = entry.strip()
    if not entry:
        continue
    if entry.startswith(": "):
        header_count += 1
        sub_category = entry
        category = "Semantic" if header_count <= 5 else "Syntactic"
    else:
        if sub_category is None or len(entry.split()) != 4:
            raise ValueError(f"Malformed analogy at line {line_number}: {entry!r}")
        questions.append(entry)
        categories.append(category)
        sub_categories.append(sub_category)
assert header_count == 14 and len(questions) == 19544
print(f"Parsed {len(questions):,} analogies across {header_count} sub-categories.")


In [ ]:
# Create the dataframe
df = pd.DataFrame(
    {
        "Question": questions,
        "Category": categories,
        "SubCategory": sub_categories,
    }
)

In [ ]:
df.head()

In [ ]:
df.to_csv(f"{file_name}.csv", index=False)

## Part II: Use pre-trained word embeddings
- After finish Part I, you can run Part II code blocks only.

In [ ]:
import pandas as pd
import numpy as np
import gensim.downloader
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

In [ ]:
data = pd.read_csv("questions-words.csv")

In [ ]:
MODEL_NAME = "glove-wiki-gigaword-100"
# You can try other models.
# https://radimrehurek.com/gensim/models/word2vec.html#pretrained-models

# Load the pre-trained model (using GloVe vectors here)
model = gensim.downloader.load(MODEL_NAME)
print("The Gensim model loaded successfully!")

In [ ]:
# Do predictions and preserve the gold answers (word_D)
preds = []
golds = []

# I implement the analogy predictions with the help of OpenAI Codex.
prediction_values, gold_values, glove_detail = analogy_predictions(model, data, "glove")
prediction_iterator = iter(zip(prediction_values, gold_values))

for analogy in tqdm(data["Question"]):
      # TODO2: Write your code here to use pre-trained word embeddings for getting predictions of the analogy task.
      # You should also preserve the gold answers during iterations for evaluations later.
      """ Hints
      # Unpack the analogy (e.g., "man", "woman", "king", "queen")
      # Perform vector arithmetic: word_b + word_c - word_a should be close to word_d
      # Source: https://github.com/piskvorky/gensim/blob/develop/gensim/models/keyedvectors.py#L776
      # Mikolov et al., 2013: big - biggest and small - smallest
      # Mikolov et al., 2013: X = vector(”biggest”) − vector(”big”) + vector(”small”).
      """
      prediction, gold = next(prediction_iterator)
      preds.append(prediction)
      golds.append(gold)

pretrained_vectors = model


In [ ]:
# Perform evaluations. You do not need to modify this block!!

def calculate_accuracy(gold: np.ndarray, pred: np.ndarray) -> float:
    return np.mean(gold == pred)

golds_np, preds_np = np.array(golds), np.array(preds)
data = pd.read_csv("questions-words.csv")

# Evaluation: categories
for category in data["Category"].unique():
    mask = data["Category"] == category
    golds_cat, preds_cat = golds_np[mask], preds_np[mask]
    acc_cat = calculate_accuracy(golds_cat, preds_cat)
    print(f"Category: {category}, Accuracy: {acc_cat * 100}%")

# Evaluation: sub-categories
for sub_category in data["SubCategory"].unique():
    mask = data["SubCategory"] == sub_category
    golds_subcat, preds_subcat = golds_np[mask], preds_np[mask]
    acc_subcat = calculate_accuracy(golds_subcat, preds_subcat)
    print(f"Sub-Category{sub_category}, Accuracy: {acc_subcat * 100}%")

In [ ]:
# Collect words from Google Analogy dataset
SUB_CATEGORY = ": family"

# TODO3: Plot t-SNE for the words in the SUB_CATEGORY `: family`


# I implement the t-SNE plotting with the help of OpenAI Codex.
family_figure = plot_family(pretrained_vectors, data, "GloVe: family word relationships", "family_glove.png")

plt.title("Word Relationships from Google Analogy Task")
plt.show()
plt.savefig("word_relationships.png", bbox_inches="tight")

family_figure.savefig("word_relationships.png", dpi=160, bbox_inches="tight")
plt.close("all")


### Part III: Train your own word embeddings

### Get the latest English Wikipedia articles and do sampling.
- Usually, we start from Wikipedia dump (https://dumps.wikimedia.org/enwiki/latest/enwiki-latest-pages-articles.xml.bz2). However, the downloading step will take very long. Also, the cleaning step for the Wikipedia corpus ([`gensim.corpora.wikicorpus.WikiCorpus`](https://radimrehurek.com/gensim/corpora/wikicorpus.html#gensim.corpora.wikicorpus.WikiCorpus)) will take much time. Therefore, we provide cleaned files for you.

In [ ]:
# Download the split Wikipedia files
# Each file contain 562365 lines (articles).
!gdown --id 1jiu9E1NalT2Y8EIuWNa1xf2Tw1f1XuGd -O wiki_texts_part_0.txt.gz
!gdown --id 1ABblLRd9HXdXvaNv8H9fFq984bhnowoG -O wiki_texts_part_1.txt.gz
!gdown --id 1z2VFNhpPvCejTP5zyejzKj5YjI_Bn42M -O wiki_texts_part_2.txt.gz
!gdown --id 1VKjded9BxADRhIoCzXy_W8uzVOTWIf0g -O wiki_texts_part_3.txt.gz
!gdown --id 16mBeG26m9LzHXdPe8UrijUIc6sHxhknz -O wiki_texts_part_4.txt.gz

In [ ]:
# Download the split Wikipedia files
# Each file contain 562365 lines (articles), except the last file.
!gdown --id 17JFvxOH-kc-VmvGkhG7p3iSZSpsWdgJI -O wiki_texts_part_5.txt.gz
!gdown --id 19IvB2vOJRGlrYulnTXlZECR8zT5v550P -O wiki_texts_part_6.txt.gz
!gdown --id 1sjwO8A2SDOKruv6-8NEq7pEIuQ50ygVV -O wiki_texts_part_7.txt.gz
!gdown --id 1s7xKWJmyk98Jbq6Fi1scrHy7fr_ellUX -O wiki_texts_part_8.txt.gz
!gdown --id 17eQXcrvY1cfpKelLbP2BhQKrljnFNykr -O wiki_texts_part_9.txt.gz
!gdown --id 1J5TAN6bNBiSgTIYiPwzmABvGhAF58h62 -O wiki_texts_part_10.txt.gz

In [ ]:
# Extract the downloaded wiki_texts_parts files.
!gunzip -k wiki_texts_part_*.gz

In [ ]:
# Combine the extracted wiki_texts_parts files.
!cat wiki_texts_part_*.txt > wiki_texts_combined.txt

In [ ]:
# Check the first ten lines of the combined file
!head -n 10 wiki_texts_combined.txt

Please note that we used the default parameters of [`gensim.corpora.wikicorpus.WikiCorpus`](https://radimrehurek.com/gensim/corpora/wikicorpus.html#gensim.corpora.wikicorpus.WikiCorpus) for cleaning the Wiki raw file. Thus, words with one character were discarded.

In [ ]:
# Now you need to do sampling because the corpus is too big.
# You can further perform analysis with a greater sampling ratio.

import random

wiki_txt_path = "wiki_texts_combined.txt"
# wiki_texts_combined.txt is a text file separated by linebreaks (\n).
# Each row in wiki_texts_combined.txt indicates a Wikipedia article.

# I implement the sampling with the help of OpenAI Codex.
output_path = str(CACHE / "wiki_sample_20.pending.txt")

with open(wiki_txt_path, "r", encoding="utf-8") as f:
    with open(output_path, "w", encoding="utf-8") as output_file:
    # TODO4: Sample `20%` Wikipedia articles
    # Write your code here
        total_articles = sum(1 for _ in f)
        f.seek(0)
        remaining = math.floor(total_articles * 0.20)
        sample_count = remaining
        sample_rng = random.Random(SEED)
        for article_index, article in enumerate(f):
            if sample_rng.randrange(total_articles - article_index) < remaining:
                output_file.write(article)
                remaining -= 1
        assert remaining == 0
        print(f"20% sample: {sample_count:,} of {total_articles:,} articles")
        wiki_sample_metadata = {"total_articles": total_articles, "sample_articles": sample_count,
                                "fraction": 0.20, "seed": SEED}
        (RESULTS / "wiki_20_sampling.json").write_text(json.dumps(wiki_sample_metadata, indent=2))

# Commit the sample after closing the template file handles.
import filecmp
sample_target = CACHE / "wiki_sample_20.txt"
if sample_target.exists() and filecmp.cmp(output_path, sample_target, shallow=False):
    Path(output_path).unlink()
else:
    Path(output_path).replace(sample_target)
output_path = str(sample_target)


In [ ]:
# TODO5: Train your own word embeddings with the sampled articles
# https://radimrehurek.com/gensim/models/word2vec.html#gensim.models.word2vec.Word2Vec
# Hint: You should perform some pre-processing before training.
model, wiki_20_metadata = train_embeddings(output_path, "wiki_20")
wiki_vectors = model.wv
print(json.dumps(wiki_20_metadata, indent=2))


In [ ]:
data = pd.read_csv("questions-words.csv")

In [ ]:
# Do predictions and preserve the gold answers (word_D)
preds = []
golds = []

# I implement the analogy predictions with the help of OpenAI Codex.
prediction_values, gold_values, wiki_20_detail = analogy_predictions(model.wv, data, "wiki_20")
prediction_iterator = iter(zip(prediction_values, gold_values))

for analogy in tqdm(data["Question"]):
      # TODO6: Write your code here to use your trained word embeddings for getting predictions of the analogy task.
      # You should also preserve the gold answers during iterations for evaluations later.
      """ Hints
      # Unpack the analogy (e.g., "man", "woman", "king", "queen")
      # Perform vector arithmetic: word_b + word_c - word_a should be close to word_d
      # Source: https://github.com/piskvorky/gensim/blob/develop/gensim/models/keyedvectors.py#L776
      # Mikolov et al., 2013: big - biggest and small - smallest
      # Mikolov et al., 2013: X = vector(”biggest”) − vector(”big”) + vector(”small”).
      """
      prediction, gold = next(prediction_iterator)
      preds.append(prediction)
      golds.append(gold)


In [ ]:
# Collect words from Google Analogy dataset
SUB_CATEGORY = ": family"

# TODO7: Plot t-SNE for the words in the SUB_CATEGORY `: family`


# I implement the t-SNE plotting with the help of OpenAI Codex.
family_figure = plot_family(model.wv, data, "Word2Vec (20% Wikipedia): family word relationships", "family_wiki_20.png")

plt.title("Word Relationships from Google Analogy Task")
plt.show()
plt.savefig("word_relationships.png", bbox_inches="tight")

family_figure.savefig("word_relationships.png", dpi=160, bbox_inches="tight")
plt.close("all")


Running environment: 
- System: Windows 11
- CPU : AMD Ryzen 9 5900HS CPU

Python version: 3.13.5

##### 1. Which embedding model do you use? What are the pre-processing steps? What are the hyperparameter settings? (5%)

Answer: I used the pre-trained GloVe Wiki + Gigaword model as baseline. I trained the word embeddings using Gensim Word2Vec with CBOW and negative sampling.

For Word2Vec, I used vector_size=100, window=5, min_count=5, sg=0, negative=5, epochs=5, workers=8, and seed=42, with maximum vocabulary size of 200,000 words. I used the same settings for all datasets.

For preprocessing, I converted all words to lowercase and removed tokens containing numbers, punctuation, or non-ASCII characters. I kept stop words and different word forms because they may be useful for syntactic analogy questions. For the Brown corpus, I also removed the part-of-speech tags before applying the same preprocessing.

For evaluation, I used the formula B + C − A to predict the missing word based on cosine similarity.

##### 2. What will the performance be like if you sample 5%, 10% and 20% of wiki text in TODO4? (10%, 3% for each)

Answer:
The results are shown below:

| Dataset | Articles | Accuracy | Training Time |
| --- | --- | --- | --- |
| Wiki 5% | 281,182 | 52.92% | 294.8s |
| Wiki 10% | 562,365 | 55.54% | 600.7s |
| Wiki 20% | 1,124,731 | 56.70% | 1205.9s |

From the results, increasing training data improves model's accuracy. The accuracy increased by 2.62% from Wiki 5% to Wiki 10%, and another 1.16% from Wiki 10% to Wiki 20%. This shows that using more training data helps Word2Vec learn better word relationships. However, the improvement becomes smaller as the dataset grows which require longer training time.

Overall, Wiki 20% has the highest accuracy at 56.70%, but also required the longest training time. 

##### 3. What is the performance for different categories or sub-categories when trained on different corpora? (15%)

3.1. Present your results. (5%)

Answer: I compared GloVe, three Wiki models (5%, 10%, and 20%), and the Brown corpus model.
The results for semantic and syntactic categories are shown below:

| Group | GloVe | Wiki 5% | Wiki 10% | Wiki 20% | Brown |
| --- | --- | --- | --- | --- | --- |
| Semantic | 65.34% | 58.42% | 61.31% | 63.39% | 0.32% |
| Syntactic | 61.26% | 48.36% | 50.74% | 51.15% | 1.33% |

Sub-category results:

| Group | GloVe | Wiki 5% | Wiki 10% | Wiki 20% | Brown |
| --- | --- | --- | --- | --- | --- |
| : capital-common-countries | 93.87% | 80.63% | 79.25% | 78.85% | 0.00% |
| : capital-world | 88.95% | 71.11% | 74.25% | 77.23% | 0.00% |
| : city-in-state | 30.81% | 43.78% | 46.94% | 47.06% | 0.04% |
| : currency | 14.20% | 7.97% | 12.24% | 15.59% | 0.00% |
| : family | 81.62% | 80.43% | 81.82% | 85.57% | 5.34% |
| : gram1-adjective-to-adverb | 24.40% | 18.55% | 20.16% | 20.46% | 0.10% |
| : gram2-opposite | 20.07% | 11.95% | 14.78% | 17.61% | 0.00% |
| : gram3-comparative | 79.13% | 74.47% | 73.20% | 72.45% | 5.78% |
| : gram4-superlative | 54.28% | 32.62% | 40.02% | 39.93% | 0.00% |
| : gram5-present-participle | 69.51% | 38.07% | 42.05% | 44.32% | 0.95% |
| : gram6-nationality-adjective | 87.87% | 78.55% | 79.74% | 80.61% | 0.06% |
| : gram7-past-tense | 55.45% | 48.72% | 50.45% | 51.73% | 2.69% |
| : gram8-plural | 72.00% | 51.58% | 52.48% | 50.53% | 0.53% |
| : gram9-plural-verbs | 58.39% | 48.05% | 53.79% | 53.33% | 0.46% |

Overall, GloVe achieved the highest accuracy in both semantic and syntactic categories. Wiki 20% performed better than the smaller models, while Brown had the lowest accuracy.

3.2. Introduce the corpus you selected and explain what are the differences between the Wikipedia corpus and your corpus. (including data size, topic difference, structural difference … ) (5%)

Answer: I selected the Brown corpus which contains 500 American English texts from different categories, such as news, fiction, reviews, and religion. It contains about 981,716 cleaned tokens, while the 20% Wikipedia dataset contains 666.5 million tokens.

Wikipedia contains encyclopedic articles covering many topics, such as countries, cities, history, and technology. In comparison, Brown contains a smaller collection of general English texts from different writing styles. Brown also includes part-of-speech tags, which I removed during preprocessing.

I used this corpus because of the differences in dataset size, content, and structure. These differences can help see how much the performance in word relationships are affected.

3.3. Explain why the performance increases or decreases. (5%)

Answer: Wiki 20% model achieved 56.70% overall accuracy, while Brown achieved only 0.87%.

One main reason is the difference in dataset size. Wikipedia contains much more training data that helps learn more word relationships. Brown also has much lower vocabulary coverage (32.26%) compared with Wikipedia (99.08%). This results in many analogy questions' words that Brown does not recognize.

##### 4. Select a few words and use their embeddings to retrieve the five most similar words. What do you observe? (10%)

Answer: I selected king, queen, computer, music, and city.

| Model | Word | Neighbors (cosine) |
| --- | --- | --- |
| GloVe | king | prince (0.768); queen (0.751); son (0.702); brother (0.699); monarch (0.698) |
| GloVe | queen | princess (0.795); king (0.751); elizabeth (0.736); royal (0.707); lady (0.704) |
| GloVe | computer | computers (0.875); software (0.837); technology (0.764); pc (0.737); hardware (0.729) |
| GloVe | music | musical (0.813); songs (0.798); dance (0.790); pop (0.786); recording (0.765) |
| GloVe | city | town (0.826); cities (0.776); where (0.755); area (0.746); downtown (0.744) |
| Wiki 20% | king | prince (0.806); unready (0.742); conqueror (0.727); ethelred (0.720); monarch (0.713) |
| Wiki 20% | queen | princess (0.769); king (0.703); katwe (0.695); coronation (0.687); sirikit (0.680) |
| Wiki 20% | computer | computing (0.767); software (0.767); computers (0.765); mainframe (0.732); microcomputer (0.718) |
| Wiki 20% | music | jazz (0.744); musical (0.724); dance (0.702); hiphop (0.665); pop (0.653) |
| Wiki 20% | city | town (0.740); downtown (0.644); lenexa (0.641); municipal (0.639); suburb (0.629) |
| Brown | king | boxcar (0.923); news (0.906); spanish (0.895); crowd (0.893); seminary (0.889) |
| Brown | queen | youngest (0.961); leonard (0.959); harris (0.952); senator (0.950); milwaukee (0.950) |
| Brown | computer | address (0.938); pencil (0.926); picturesque (0.926); autocollimator (0.924); vicious (0.924) |
| Brown | music | clarity (0.897); suffering (0.874); stake (0.865); taste (0.858); coherent (0.857) |
| Brown | city | washington (0.852); chicago (0.849); company (0.849); post (0.842); bank (0.838) |

From the results, I observed that GloVe and Wiki 20% generally produce more meaningful word relationships than Brown. For example, both GloVe and Wiki 20% return technical words such as software and computers for computer. For king and queen, GloVe mostly gives royal or family-related words, while Wiki 20% also includes historical names and terms. For music, GloVe gives words such as songs and recording, while Wiki 20% gives music genres such as jazz and hiphop. Brown seems to perform the worst, giving mostly unrelated words.

Overall, larger training corpora tend to produce more meaningful word embeddings because they contain more examples of different word relationships. The results also show that different corpora can learn different associations for the same word.

##### 5. … Anything that can strengthen your report. (5%)

Answer: To further analyze the models, I compared their average accuracy across the 14 subcategories.

| Model | Average Subcategory Accuracy |
| --- | --- |
| GloVe | 59.32% |
| Wiki 5% | 49.03% |
| Wiki 10% | 51.51% |
| Wiki 20% | 52.52% |
| Brown | 1.14% |

The results unsuprisingly show that GloVe performs the best overall, followed by Wiki 20%, while Brown has the lowest accuracy.

In the Wiki 20% model, some of the words are not entirely wrong. For example, when the expected answer was Iraq, the model predicted Syria, which is similar but not quite the correct answer.

##### 6. Generative AI Usage

Answer: OpenAI Codex assisted with the TODO implementations, portable data-loading code, experiment code, and gathering the necessary results for the report. I trained the model, verified the results after Codex's implementation, and wrote this report from the results of the models with the guidance of OpenAI Codex.


## Code For 3. What is the performance for different categories or sub-categories when trained on different corpora? (15%)

In [ ]:
# Write your code here
# I implement the evaluation with the help of OpenAI Codex.

from IPython.display import display, Markdown

experiment_details = {"GloVe": glove_detail, "Wiki 20%": wiki_20_detail}
experiment_metadata = {"Wiki 20%": wiki_20_metadata}
for ratio in (0.05, 0.10):
    label = f"wiki_{round(ratio * 100)}"
    sampled_path = CACHE / f"{label}_sample.txt"
    sampling_stats = sample_articles(wiki_txt_path, sampled_path, ratio)
    experiment_model, metadata = train_embeddings(sampled_path, label)
    _, _, detail = analogy_predictions(experiment_model.wv, data, label)
    experiment_details[f"Wiki {ratio:.0%}"] = detail
    experiment_metadata[f"Wiki {ratio:.0%}"] = metadata
    del experiment_model

brown_path = prepare_brown()
brown_model, brown_metadata = train_embeddings(brown_path, "brown")
_, _, brown_detail = analogy_predictions(brown_model.wv, data, "brown")
experiment_details["Brown"] = brown_detail
experiment_metadata["Brown"] = brown_metadata

score_frames = []
for label, detail in experiment_details.items():
    summary = summarize_scores(detail)
    summary.insert(0, "Model", label)
    score_frames.append(summary)
comparison = pd.concat(score_frames, ignore_index=True)
comparison.to_csv(RESULTS / "comparison.csv", index=False)
display(comparison)
neighbors = nearest_words({"GloVe": pretrained_vectors, "Wiki 20%": wiki_vectors, "Brown": brown_model.wv})
display(neighbors)

overall = comparison.loc[comparison.Level == "Overall"].copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(overall.Model, overall.Accuracy * 100, color="#245b87")
axes[0].set_ylabel("Accuracy (%) over all 19,544 questions")
axes[1].bar(overall.Model, overall.Coverage * 100, color="#56866b")
axes[1].set_ylabel("Four-word vocabulary coverage (%)")
for axis in axes:
    axis.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(RESULTS / "model_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

environment = {"system": platform.platform(), "cpu": platform.processor(),
               "logical_cpus": os.cpu_count(), "python": sys.version,
               "packages": {name: __import__(name).__version__ for name in
                            ("gensim", "numpy", "pandas", "scipy", "sklearn", "matplotlib")}}
(RESULTS / "environment.json").write_text(json.dumps(environment, indent=2))
(RESULTS / "experiment_metadata.json").write_text(json.dumps(experiment_metadata, indent=2))
print("Completed all experiments. Report tables and neighbor lists contain measured outputs.")
